# NLU HOSTAGE — ai_1_nlu_v3_600

Notebook ini hanya memakai modul training bersama. Semua cell legacy telah dihapus.


In [ ]:
from pathlib import Path
DATASET_FILENAME = "dataset_final3.csv"
NOTEBOOK_FOLDER = "ai_2_dataset_baru"
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != NOTEBOOK_FOLDER:
    candidate = NOTEBOOK_DIR / NOTEBOOK_FOLDER
    if candidate.is_dir():
        NOTEBOOK_DIR = candidate
DATASET_PATH = NOTEBOOK_DIR / "data" / DATASET_FILENAME
print(f"Dataset aktif: {DATASET_PATH}")


In [ ]:
# MODUL BERSAMA: EDA + SVM + SVM TUNING + NB + NB TUNING + TRANSFORMER
from pathlib import Path
import sys

PROJECT_ROOT = Path(NOTEBOOK_DIR).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.nlu_eda import run_nlu_eda
from modules.nlu_training import (
    predict_intent as _predict_intent_shared,
    predict_transformer_intent as _predict_transformer_intent_shared,
    run_all_nlu_models,
    train_naive_bayes,
    train_naive_bayes_tuned,
    train_svm,
    train_svm_tuned,
    train_transformer,
)

MODEL_DIR = Path(NOTEBOOK_DIR) / "models"
# Kebijakan proyek: SVM/NB pada CPU, Transformer pada GPU CUDA.
CLASSICAL_DEVICE = "cpu"
TRANSFORMER_DEVICE = "cuda"
# Fine-tuning: seluruh bobot IndoBERT dilatih. Epoch adalah batas maksimum.
TRANSFORMER_EPOCHS = 12
TRANSFORMER_OPTIONS = {
    "learning_rates": (1e-5, 2e-5, 3e-5),  # pilih hanya dari validation macro-F1
    "validation_size": 0.15,  # train/validation/test sekitar 65/15/20
    "batch_size": 8,
    "eval_batch_size": 16,
    "gradient_accumulation_steps": 2,  # effective batch 16 pada satu GPU
    "max_length": 128,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "linear",
    "dropout": 0.1,
    "label_smoothing_factor": 0.0,
    "max_grad_norm": 1.0,
    "early_stopping_patience": 3,
    "early_stopping_threshold": 0.001,
    "gradient_checkpointing": False,  # aktifkan jika memori GPU tidak cukup
}


def run_eda(plot=True):
    return run_nlu_eda(DATASET_PATH, plot=plot)

def train_svm_model():
    return train_svm(DATASET_PATH, MODEL_DIR)

def train_svm_tuned_model():
    return train_svm_tuned(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_model():
    return train_naive_bayes(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_tuned_model():
    return train_naive_bayes_tuned(DATASET_PATH, MODEL_DIR)

def train_transformer_model(epochs=None, **overrides):
    options = {**TRANSFORMER_OPTIONS, **overrides}
    # Untuk satu LR: train_transformer_model(learning_rates=None, learning_rate=2e-5)
    return train_transformer(
        DATASET_PATH, MODEL_DIR,
        epochs=TRANSFORMER_EPOCHS if epochs is None else epochs,
        device=TRANSFORMER_DEVICE, **options,
    )

def predict_intent(text, model_filename=None):
    return _predict_intent_shared(text, MODEL_DIR, model_filename)

def predict_transformer_intent(text):
    return _predict_transformer_intent_shared(text, MODEL_DIR, device=TRANSFORMER_DEVICE)

# Tiga intent dataset Janice; dipakai juga untuk evaluasi setelah training.
HOSTAGE_TEST_CASES = [
    ("offend", "B kena Gag Order ketika mulai ditanya alibinya, menurutku itu pola Hitman."),
    ("defend", "Aku bukan Hitman. Tuduhan itu tidak punya bukti dari chat publik."),
    ("defend", "Aku Spy dan semalam Guard Raka, jadi jangan curigai dia."),
    ("neutral", "Stalker, kamu Peek siapa semalam dan apa hasil yang kamu lihat?"),
    ("offend", "Jangan hanya fokus ke aku; cek D yang ceritanya berubah setiap ditanya."),
    ("offend", "Vote C saja, dia paling diuntungkan ketika seseorang mendadak diam."),
    ("defend", "Klaimku Civilian, jadi aku memang tidak memiliki aksi malam."),
    ("neutral", "Malam ini chat terkunci, kita lanjut diskusi setelah fase pagi."),
    ("offend", "A selalu menghindar saat kita tanya kenapa dia menuduh tanpa bukti."),
    ("defend", "Diamku bukan pengakuan; aku masih bisa menjelaskan alibiku di fase siang."),
]

def run_hostage_test_suite(model_filename=None):
    correct = 0
    for expected, chat in HOSTAGE_TEST_CASES:
        predicted, confidence = predict_intent(chat, model_filename)
        correct += predicted == expected
        print(f"{expected:12} | prediksi={predicted:12} | confidence={confidence:6.2f}% | {chat}")
    print(f"\nCocok: {correct}/{len(HOSTAGE_TEST_CASES)}")

print("Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),")
print("train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().")
print("Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.")
print("Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.")


In [ ]:
run_eda()

In [ ]:
# Muat ulang modul agar kernel memakai perubahan kode terbaru.
import importlib
import modules.nlu_training as nlu_training
nlu_training = importlib.reload(nlu_training)
run_all_nlu_models = nlu_training.run_all_nlu_models

# Jalankan baseline CPU dan fine-tuning IndoBERT CUDA; tuning SVM/NB opsional.
RUN_TRANSFORMER = True
RUN_TUNING = False  # khusus grid search SVM/NB; IndoBERT memakai TRANSFORMER_OPTIONS
artifacts, hasil_training, hasil_manual_test = run_all_nlu_models(
    DATASET_PATH, MODEL_DIR,
    run_transformer=RUN_TRANSFORMER,
    run_tuning=RUN_TUNING,
    transformer_epochs=TRANSFORMER_EPOCHS,
    transformer_device=TRANSFORMER_DEVICE,
    transformer_options=TRANSFORMER_OPTIONS,
    benchmark_tests=HOSTAGE_TEST_CASES,
)
print('RINGKASAN EVALUASI HOLDOUT:')
display(hasil_training)
print('RINGKASAN 10 CHAT UJI:')
display(hasil_manual_test)

# Riwayat loss dan validation macro-F1 per epoch; test tidak dipakai memilih model.
if "IndoBERT Transformer" in artifacts:
    import pandas as pd
    transformer_summary = artifacts["IndoBERT Transformer"]["artifact"]["training_summary"]
    display(pd.DataFrame([
        {k: v for k, v in trial.items() if k not in {"history", "best_checkpoint"}}
        for trial in transformer_summary["trials"]
    ]))
    for trial in transformer_summary["trials"]:
        print(f"Learning rate: {trial['learning_rate']:g}")
        display(pd.DataFrame(trial["history"]))


In [3]:
# TUNING SVM DAN NAIVE BAYES
# Cell ini dapat dijalankan langsung dari folder proyek atau subfoldernya.
from pathlib import Path
import sys
import importlib
from time import perf_counter

_current_dir = Path.cwd().resolve()
PROJECT_ROOT = next((
    folder for folder in (_current_dir, *_current_dir.parents)
    if (folder / "modules" / "nlu_training.py").is_file()
    and (folder / "ai_2_dataset_baru" / "data" / "dataset_final3.csv").is_file()
), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Folder proyek tidak ditemukan. Buka notebook dari workspace prethesis."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_DIR = PROJECT_ROOT / "ai_2_dataset_baru"
DATASET_PATH = NOTEBOOK_DIR / "data" / "dataset_final3.csv"
MODEL_DIR = NOTEBOOK_DIR / "models"
HOSTAGE_TEST_CASES = [
    ("offend", "B kena Gag Order ketika mulai ditanya alibinya, menurutku itu pola Hitman."),
    ("defend", "Aku bukan Hitman. Tuduhan itu tidak punya bukti dari chat publik."),
    ("defend", "Aku Spy dan semalam Guard Raka, jadi jangan curigai dia."),
    ("neutral", "Stalker, kamu Peek siapa semalam dan apa hasil yang kamu lihat?"),
    ("offend", "Jangan hanya fokus ke aku; cek D yang ceritanya berubah setiap ditanya."),
    ("offend", "Vote C saja, dia paling diuntungkan ketika seseorang mendadak diam."),
    ("defend", "Klaimku Civilian, jadi aku memang tidak memiliki aksi malam."),
    ("neutral", "Malam ini chat terkunci, kita lanjut diskusi setelah fase pagi."),
    ("offend", "A selalu menghindar saat kita tanya kenapa dia menuduh tanpa bukti."),
    ("defend", "Diamku bukan pengakuan; aku masih bisa menjelaskan alibiku di fase siang."),
]

import pandas as pd
from IPython.display import display
import modules.nlu_training as nlu_training

nlu_training = importlib.reload(nlu_training)
artifacts_tuning = {}
ringkasan_tuning = []
chat_tuning = []

for nama, trainer in [
    ("SVM tuned", nlu_training.train_svm_tuned),
    ("Naive Bayes tuned", nlu_training.train_naive_bayes_tuned),
]:
    print(f"\nMENJALANKAN: {nama}", flush=True)
    mulai = perf_counter()
    hasil = trainer(DATASET_PATH, MODEL_DIR)
    durasi = perf_counter() - mulai
    artifacts_tuning[nama] = hasil
    metrik = hasil["metrics"]
    ringkasan_tuning.append({
        "model": nama,
        "accuracy_holdout": metrik["accuracy"],
        "macro_f1_holdout": metrik["f1_macro"],
        "weighted_f1_holdout": metrik["f1_weighted"],
        "macro_f1_cv": hasil["cv_f1_macro"],
        "waktu_detik": round(durasi, 1),
    })
    print("Parameter terbaik:", hasil["best_params"])
    for expected, chat in HOSTAGE_TEST_CASES:
        predicted, confidence = nlu_training.predict_intent(
            chat, MODEL_DIR, filename=Path(hasil["model_path"]).name,
        )
        chat_tuning.append({
            "model": nama, "chat": chat, "expected": expected,
            "predicted": predicted, "confidence_persen": confidence,
            "benar": predicted == expected,
        })

hasil_training_tuning = pd.DataFrame(ringkasan_tuning)
hasil_manual_test_tuning = pd.DataFrame(chat_tuning)
print("\nRINGKASAN EVALUASI HOLDOUT TUNING:")
display(hasil_training_tuning)
print("\nHASIL CHAT UJI TUNING:")
display(hasil_manual_test_tuning)
display(hasil_manual_test_tuning.groupby("model")["benar"].agg(
    jumlah_benar="sum", jumlah_chat="count", akurasi="mean",
).reset_index())



MENJALANKAN: SVM tuned
Memulai SVM Grid Search pada train set...
Fitting 5 folds for each of 720 candidates, totalling 3600 fits
Parameter SVM terbaik: {'svm__C': 1.5, 'svm__class_weight': 'balanced', 'tfidf__analyzer': 'word', 'tfidf__max_df': 0.95, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 3), 'tfidf__sublinear_tf': False}
Macro-F1 CV terbaik: 0.8420

--- Evaluasi SVM tuned (holdout test set) ---
Accuracy    : 0.8625
Macro F1    : 0.8395
Weighted F1 : 0.8627
              precision    recall  f1-score   support

      defend       0.74      0.76      0.75       188
     neutral       0.87      0.88      0.87       419
      offend       0.90      0.89      0.89       513

    accuracy                           0.86      1120
   macro avg       0.84      0.84      0.84      1120
weighted avg       0.86      0.86      0.86      1120

Model tersimpan: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_2_dataset_baru\models\intent_classifier_svm_tuned.pkl
Parameter terbaik: {'

,model,accuracy_holdout,macro_f1_holdout,weighted_f1_holdout,macro_f1_cv,waktu_detik
0,SVM tuned,0.8625,0.8395,0.8627,0.8420,195.2
1,Naive Bayes tuned,0.8455,0.8169,0.8458,0.8171,383.5



HASIL CHAT UJI TUNING:


,model,chat,expected,predicted,confidence_persen,benar
0,SVM tuned,B kena Gag Order ketika mulai ditanya alibinya...,offend,offend,98.42,True
1,SVM tuned,Aku bukan Hitman. Tuduhan itu tidak punya bukt...,defend,defend,95.82,True
2,SVM tuned,"Aku Spy dan semalam Guard Raka, jadi jangan cu...",defend,defend,85.40,True
3,SVM tuned,"Stalker, kamu Peek siapa semalam dan apa hasil...",neutral,neutral,99.04,True
4,SVM tuned,Jangan hanya fokus ke aku; cek D yang ceritany...,offend,offend,70.46,True
5,SVM tuned,"Vote C saja, dia paling diuntungkan ketika ses...",offend,offend,99.23,True
6,SVM tuned,"Klaimku Civilian, jadi aku memang tidak memili...",defend,neutral,68.89,False
7,SVM tuned,"Malam ini chat terkunci, kita lanjut diskusi s...",neutral,neutral,87.81,True
8,SVM tuned,A selalu menghindar saat kita tanya kenapa dia...,offend,offend,99.65,True
9,SVM tuned,Diamku bukan pengakuan; aku masih bisa menjela...,defend,neutral,54.73,False


,model,jumlah_benar,jumlah_chat,akurasi
0,Naive Bayes tuned,7,10,0.7
1,SVM tuned,8,10,0.8


In [2]:
# TUNING OPTUNA: NAIVE BAYES DAN SVM
# Cell ini dapat dijalankan langsung dari folder proyek atau subfoldernya.
from pathlib import Path
import sys
import importlib
from time import perf_counter

_current_dir = Path.cwd().resolve()
PROJECT_ROOT = next((
    folder for folder in (_current_dir, *_current_dir.parents)
    if (folder / "modules" / "nlu_training.py").is_file()
    and (folder / "ai_2_dataset_baru" / "data" / "dataset_final3.csv").is_file()
), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Folder proyek tidak ditemukan. Buka notebook dari workspace prethesis."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_DIR = PROJECT_ROOT / "ai_2_dataset_baru"
DATASET_PATH = NOTEBOOK_DIR / "data" / "dataset_final3.csv"
MODEL_DIR = NOTEBOOK_DIR / "models"
HOSTAGE_TEST_CASES = [
    ("offend", "B kena Gag Order ketika mulai ditanya alibinya, menurutku itu pola Hitman."),
    ("defend", "Aku bukan Hitman. Tuduhan itu tidak punya bukti dari chat publik."),
    ("defend", "Aku Spy dan semalam Guard Raka, jadi jangan curigai dia."),
    ("neutral", "Stalker, kamu Peek siapa semalam dan apa hasil yang kamu lihat?"),
    ("offend", "Jangan hanya fokus ke aku; cek D yang ceritanya berubah setiap ditanya."),
    ("offend", "Vote C saja, dia paling diuntungkan ketika seseorang mendadak diam."),
    ("defend", "Klaimku Civilian, jadi aku memang tidak memiliki aksi malam."),
    ("neutral", "Malam ini chat terkunci, kita lanjut diskusi setelah fase pagi."),
    ("offend", "A selalu menghindar saat kita tanya kenapa dia menuduh tanpa bukti."),
    ("defend", "Diamku bukan pengakuan; aku masih bisa menjelaskan alibiku di fase siang."),
]

import pandas as pd
from IPython.display import display
import modules.nlu_training as nlu_training

nlu_training = importlib.reload(nlu_training)
OPTUNA_TRIALS = 50  # jumlah trial per model; ubah sesuai waktu eksperimen
OPTUNA_CV_FOLDS = 5
OPTUNA_TIMEOUT = None  # batas pencarian per model (detik), atau None
OPTUNA_N_JOBS = -1  # paralel fold CV; trial Optuna tetap berurutan

artifacts_optuna = {}
ringkasan_optuna = []
chat_optuna = []

for nama, trainer in [
    ("Naive Bayes Optuna", nlu_training.train_naive_bayes_optuna),
    ("SVM Optuna", nlu_training.train_svm_optuna),
]:
    print(f"\nMENJALANKAN: {nama}", flush=True)
    mulai = perf_counter()
    hasil = trainer(
        DATASET_PATH, MODEL_DIR, n_trials=OPTUNA_TRIALS,
        cv_folds=OPTUNA_CV_FOLDS, timeout=OPTUNA_TIMEOUT, n_jobs=OPTUNA_N_JOBS,
    )
    durasi = perf_counter() - mulai
    artifacts_optuna[nama] = hasil
    metrik = hasil["metrics"]
    ringkasan_optuna.append({
        "model": nama,
        "accuracy_holdout": metrik["accuracy"],
        "macro_f1_holdout": metrik["f1_macro"],
        "weighted_f1_holdout": metrik["f1_weighted"],
        "macro_f1_cv": hasil["cv_f1_macro"],
        "std_f1_cv": hasil["cv_std"],
        "jumlah_trial": len(hasil["study"].trials),
        "waktu_detik": round(durasi, 1),
    })
    print("Parameter terbaik:", hasil["best_params"])
    print("Riwayat eksperimen:", hasil["run_dir"])
    for expected, chat in HOSTAGE_TEST_CASES:
        predicted, confidence = nlu_training.predict_intent(
            chat, MODEL_DIR, filename=Path(hasil["model_path"]).name,
        )
        chat_optuna.append({
            "model": nama, "chat": chat, "expected": expected,
            "predicted": predicted, "confidence_persen": confidence,
            "benar": predicted == expected,
        })

hasil_training_optuna = pd.DataFrame(ringkasan_optuna)
hasil_manual_test_optuna = pd.DataFrame(chat_optuna)
print("\nRINGKASAN EVALUASI HOLDOUT OPTUNA:")
display(hasil_training_optuna)
print("\nHASIL CHAT UJI OPTUNA:")
display(hasil_manual_test_optuna)
display(hasil_manual_test_optuna.groupby("model")["benar"].agg(
    jumlah_benar="sum", jumlah_chat="count", akurasi="mean",
).reset_index())



MENJALANKAN: Naive Bayes Optuna


c:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-09-24 14:10:40,320] A new study created in memory with name: nb_macro_f1


Optuna NB: 50 trial, 5 fold, macro-F1.


[I 2026-09-24 14:10:42,937] Trial 0 finished with value: 0.8166103191825929 and parameters: {'word_ngram_max': 2, 'min_df': 1, 'max_df': 0.95, 'sublinear_tf': False, 'use_idf': False, 'norm': 'l1', 'alpha': 0.030803400529839688, 'fit_prior': False}. Best is trial 0 with value: 0.8166103191825929.
[I 2026-09-24 14:10:44,981] Trial 1 finished with value: 0.8042483681730094 and parameters: {'word_ngram_max': 2, 'min_df': 1, 'max_df': 0.95, 'sublinear_tf': False, 'use_idf': False, 'norm': None, 'alpha': 0.012790390175145836, 'fit_prior': True}. Best is trial 0 with value: 0.8166103191825929.
[I 2026-09-24 14:10:47,108] Trial 2 finished with value: 0.6391091559441071 and parameters: {'word_ngram_max': 2, 'min_df': 1, 'max_df': 0.95, 'sublinear_tf': True, 'use_idf': False, 'norm': 'l1', 'alpha': 0.33456742136968215, 'fit_prior': False}. Best is trial 0 with value: 0.8166103191825929.
[I 2026-09-24 14:10:48,814] Trial 3 finished with value: 0.7853102417146627 and parameters: {'word_ngram_max'

Parameter Optuna terbaik: {'tfidf__analyzer': 'word', 'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 1, 'tfidf__max_df': 0.95, 'tfidf__sublinear_tf': False, 'tfidf__use_idf': False, 'tfidf__norm': 'l1', 'nb__alpha': 0.030803400529839688, 'nb__fit_prior': False}
Macro-F1 CV terbaik: 0.8166

--- Evaluasi NB Optuna (holdout test set) ---
Accuracy    : 0.8473
Macro F1    : 0.8171
Weighted F1 : 0.8468
              precision    recall  f1-score   support

      defend       0.71      0.69      0.70       188
     neutral       0.86      0.88      0.87       419
      offend       0.88      0.88      0.88       513

    accuracy                           0.85      1120
   macro avg       0.82      0.82      0.82      1120
weighted avg       0.85      0.85      0.85      1120

Model tersimpan: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_2_dataset_baru\models\intent_classifier_nb_optuna.pkl
Parameter terbaik: {'tfidf__analyzer': 'word', 'tfidf__ngram_range': (1, 2), 'tfidf__min_df

...

Parameter Optuna terbaik: {'tfidf__analyzer': 'word', 'tfidf__ngram_range': (1, 3), 'tfidf__min_df': 1, 'tfidf__max_df': 0.95, 'tfidf__sublinear_tf': False, 'svm__C': 1.6941686858080556, 'svm__class_weight': 'balanced'}
Macro-F1 CV terbaik: 0.8312

--- Evaluasi SVM Optuna (holdout test set) ---
Accuracy    : 0.8670
Macro F1    : 0.8380
Weighted F1 : 0.8648
              precision    recall  f1-score   support

      defend       0.81      0.67      0.73       188
     neutral       0.86      0.91      0.88       419
      offend       0.89      0.90      0.90       513

    accuracy                           0.87      1120
   macro avg       0.85      0.83      0.84      1120
weighted avg       0.87      0.87      0.86      1120

Model tersimpan: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_2_dataset_baru\models\intent_classifier_svm_optuna.pkl
Parameter terbaik: {'tfidf__analyzer': 'word', 'tfidf__ngram_range': (1, 3), 'tfidf__min_df': 1, 'tfidf__max_df': 0.95, 'tfidf__sub

,model,accuracy_holdout,macro_f1_holdout,weighted_f1_holdout,macro_f1_cv,std_f1_cv,jumlah_trial,waktu_detik
0,Naive Bayes Optuna,0.8473,0.8171,0.8468,0.8166,0.011071,50,15.6
1,SVM Optuna,0.8670,0.8380,0.8648,0.8312,0.007162,50,44.3



HASIL CHAT UJI OPTUNA:


,model,chat,expected,predicted,confidence_persen,benar
0,Naive Bayes Optuna,B kena Gag Order ketika mulai ditanya alibinya...,offend,offend,61.58,True
1,Naive Bayes Optuna,Aku bukan Hitman. Tuduhan itu tidak punya bukt...,defend,defend,47.49,True
2,Naive Bayes Optuna,"Aku Spy dan semalam Guard Raka, jadi jangan cu...",defend,neutral,40.87,False
3,Naive Bayes Optuna,"Stalker, kamu Peek siapa semalam dan apa hasil...",neutral,neutral,62.99,True
4,Naive Bayes Optuna,Jangan hanya fokus ke aku; cek D yang ceritany...,offend,offend,52.58,True
5,Naive Bayes Optuna,"Vote C saja, dia paling diuntungkan ketika ses...",offend,offend,58.61,True
6,Naive Bayes Optuna,"Klaimku Civilian, jadi aku memang tidak memili...",defend,neutral,44.48,False
7,Naive Bayes Optuna,"Malam ini chat terkunci, kita lanjut diskusi s...",neutral,neutral,53.69,True
8,Naive Bayes Optuna,A selalu menghindar saat kita tanya kenapa dia...,offend,offend,68.60,True
9,Naive Bayes Optuna,Diamku bukan pengakuan; aku masih bisa menjela...,defend,neutral,42.89,False


,model,jumlah_benar,jumlah_chat,akurasi
0,Naive Bayes Optuna,7,10,0.7
1,SVM Optuna,8,10,0.8
